# Veo 3.1 Fast First-Last-Frame-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Veo 3.1 Fast (First-Last-Frame-to-Video)** 模型 API，对应模型 ID：`veo-3.1-fast-generate-001`。

## 模型简介

Veo 3.1 Fast 是 Google 推出的视频生成模型的快速版本，本接口接收一张起始帧图片与一张结束帧图片，自动生成两帧之间的过渡视频，支持以下能力：

- **首尾帧过渡**：基于首帧和尾帧图片生成平滑过渡视频
- **音频生成**：默认开启，可自动为视频生成同步音频
- **多分辨率**：支持 720p、1080p、4k 输出
- **多时长**：支持 4s、6s、8s 三档时长

**API 端点**：`fal-ai/veo3.1/fast/first-last-frame-to-video`

## 1. 环境准备
首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 首尾帧生成视频
七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：首帧 + 尾帧 + 提示词
result = run_fal_task(
    "fal-ai/veo3.1/fast/first-last-frame-to-video",
    arguments={
        "prompt": "A woman looks into the camera, breathes in, then exclaims energetically, \"have you guys checked out Veo3.1 First-Last-Frame-to-Video on Fal? It's incredible!\"",
        "first_frame_url": "https://storage.googleapis.com/falserverless/example_inputs/veo31-flf2v-input-1.jpeg",
        "last_frame_url": "https://storage.googleapis.com/falserverless/example_inputs/veo31-flf2v-input-2.jpeg",
        "aspect_ratio": "16:9",
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 描述视频内容的提示词，最长 20000 字符 |
| `first_frame_url` | string | *必填* | 起始帧图片 URL（也支持 base64） |
| `last_frame_url` | string | *必填* | 结束帧图片 URL（也支持 base64） |
| `duration` | string | `"8s"` | 视频时长：`"4s"`、`"6s"`、`"8s"` |
| `aspect_ratio` | string | `"auto"` | 视频宽高比：`"auto"`、`"16:9"`、`"9:16"` |
| `resolution` | string | `"720p"` | 输出分辨率：`"720p"`、`"1080p"`、`"4k"` |
| `generate_audio` | boolean | `true` | 是否同时生成音频 |
| `auto_fix` | boolean | `false` | 是否在提示词触发安全审核失败时自动改写 |
| `safety_tolerance` | string | `"4"` | 内容安全等级：`"1"`（最严格）~ `"6"`（最宽松） |
| `seed` | integer | 可选 | 随机种子，用于复现结果 |
| `negative_prompt` | string | 可选 | 负面提示词 |

## 3. 高级用法
使用更多可选参数来精细控制视频生成效果，例如指定时长、分辨率、宽高比，以及关闭音频生成。

In [ ]:
# 高级用法：1080p / 16:9 / 6 秒 / 不生成音频 / 指定 seed
result_advanced = run_fal_task(
    "fal-ai/veo3.1/fast/first-last-frame-to-video",
    arguments={
        "prompt": "A smooth cinematic transition from morning to dusk over a quiet city skyline.",
        "first_frame_url": "https://storage.googleapis.com/falserverless/example_inputs/veo31-flf2v-input-1.jpeg",
        "last_frame_url": "https://storage.googleapis.com/falserverless/example_inputs/veo31-flf2v-input-2.jpeg",
        "duration": "6s",
        "aspect_ratio": "16:9",
        "resolution": "1080p",
        "generate_audio": False,
        "negative_prompt": "blurry, low quality, distorted, watermark",
        "seed": 42,
    },
)

# 展示结果
video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 队列模式 — 手动分步调用
上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/veo3.1/fast/first-last-frame-to-video",
    arguments={
        "prompt": "A woman exclaims energetically about Veo3.1 First-Last-Frame-to-Video.",
        "first_frame_url": "https://storage.googleapis.com/falserverless/example_inputs/veo31-flf2v-input-1.jpeg",
        "last_frame_url": "https://storage.googleapis.com/falserverless/example_inputs/veo31-flf2v-input-2.jpeg",
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/veo3.1/fast/first-last-frame-to-video",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/veo3.1/fast/first-last-frame-to-video",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. Schema 参考
### Input Schema

```json
{
  "prompt": "string (必填)",
  "first_frame_url": "string (必填, 起始帧图片 URL 或 base64)",
  "last_frame_url": "string (必填, 结束帧图片 URL 或 base64)",
  "duration": "4s | 6s | 8s (可选, 默认 8s)",
  "aspect_ratio": "auto | 16:9 | 9:16 (可选, 默认 auto)",
  "resolution": "720p | 1080p | 4k (可选, 默认 720p)",
  "generate_audio": "boolean (可选, 默认 true)",
  "auto_fix": "boolean (可选, 默认 false)",
  "safety_tolerance": "1 ~ 6 (可选, 默认 4)",
  "seed": "integer (可选)",
  "negative_prompt": "string (可选)"
}
```

### Output Schema

```json
{
  "video": {
    "url": "string (视频文件 URL)"
  }
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Veo 3.1 Fast 模型页面](https://fal.ai/models/fal-ai/veo3.1/fast/first-last-frame-to-video)